# 01 · Data-Quality Assessment

Profiles the raw exports, documents every cleaning decision, and re-profiles the cleaned data so the impact of each step is measurable.

> **Synthetic data disclosure.** All data in this project is generated by `python/generate_sample_data.py` (seed 42). It is realistic in structure but does **not** describe any real company. Findings demonstrate the analytical method, not real performance.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "python"))
import pandas as pd, numpy as np
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
FIG = ROOT / "reports" / "figures"; MET = ROOT / "reports" / "metrics"
def show(name, width=900): display(Image(filename=str(FIG / name), width=width))
def pct(df, cols): return df.style.format({c: "{:.2%}" for c in cols})
import data_quality as dq

## 1. Raw data: dimensions, types and samples

In [2]:
raw = dq.load('raw')
for name, df in raw.items():
    print(f'{name:<13} {df.shape[0]:>8,} rows × {df.shape[1]} columns')
raw['sessions'].head()

sessions        35,862 rows × 10 columns
events         161,427 rows × 6 columns
transactions       976 rows × 6 columns


,session_id,visitor_id,session_start,traffic_source,marketing_medium,campaign,device_category,country,landing_page,engaged_session
0,S0026015,V0002810,2025-05-12 01:56:09,instagram,paid_social,retargeting,mobile,United States,/products/everyday-backpack,1
1,S0019982,V0006815,2025-04-12 20:30:42,IG,paid_social,spring_sale,desktop,United Kingdom,/campaign/spring-sale,1
2,S0027871,V0014537,2025-05-22T11:32:00Z,(direct),(none),NaN,desktop,United States,/products/classic-sneaker,1
3,S0016208,V0008605,2025-03-27 03:13:12,facebook,paid_social,prospecting,mobile,Germany,/products/everyday-backpack,1
4,S0028826,V0011258,2025-05-27 16:10:10,facebook,paid_social,retargeting,mobile,Canada,/collections/new-arrivals,1


In [3]:
raw['events'].head()

,event_id,session_id,event_timestamp,event_name,page_path,product_id
0,E00092362,S0020569,2025-04-15 12:40:48,page_view,/,NaN
1,E00027254,S0006120,2025-02-04 21:39:38,page_view,/,NaN
2,E00051046,S0011401,2025-03-04 23:50:40,product_view,/products/p011,P011
3,E00132365,S0029527,2025-05-30T14:47:23Z,page_view,/,NaN
4,E00020631,S0004618,2025-01-26 19:41:05,page_view,/products/p012,NaN


In [4]:
raw['transactions'].head()

,transaction_id,session_id,transaction_timestamp,revenue,currency,transaction_status
0,T0000001,S0000037,2025-01-01T07:09:24Z,60.08,USD,completed
1,T0000002,S0000049,2025-01-01 18:33:27,85.68,USD,completed
2,T0000003,S0000050,2025-01-01T03:51:01Z,26.27,USD,completed
3,T0000004,S0000083,2025-01-01 15:02:31,21.38,USD,completed
4,T0000005,S0000087,2025-01-01T22:53:43Z,67.17,USD,completed


All columns are read as text first (`dtype=str`) so that nothing is silently coerced before it has been inspected.

## 2. Raw-data profile

In [5]:
raw_report = dq.run('raw')
display(Markdown(dq.to_markdown(raw_report)))

## Data-quality profile: `raw`

| Table | Rows | Exact duplicate rows | Duplicate PKs | Unparseable timestamps | Timestamp range |
|---|---:|---:|---:|---:|---|
| sessions | 35,862 | 30 | 40 | 0 | 2025-01-01 00:02:20 → 2025-06-30 23:58:16 |
| events | 161,427 | 643 | 803 | 0 | 2025-01-01 00:02:20 → 2025-07-01 00:00:33 |
| transactions | 976 | 24 | 30 | 0 | 2025-01-01 03:51:01 → 2025-06-30 22:54:01 |

**Missing values**

- sessions: `campaign`: 20,770, `device_category`: 107, `country`: 358
- events: `product_id`: 116,124
- transactions: none

**Non-standard labels**

- sessions.`traffic_source`: 1,076 rows (e.g. ' google', '(Direct)', 'Bing', 'FACEBOOK ', 'Facebook', 'GOOGLE')
- sessions.`marketing_medium`: 717 rows (e.g. 'CPC', 'Email', 'Organic', 'Paid Social', 'Referral', 'Social')
- sessions.`device_category`: 893 rows (e.g. 'DESKTOP ', 'Desktop', 'MOBILE', 'Mobile', 'Tablet', 'iPad')
- events.`event_name`: 3,236 rows (e.g. 'ADD_TO_CART', 'Add To Cart', 'Begin Checkout', 'PAGE_VIEW', 'PURCHASE', 'Page View')
- transactions.`transaction_status`: 25 rows (e.g. 'COMPLETED', 'REFUNDED', 'test')
- transactions.`currency`: 19 rows (e.g. 'usd')

**Relationship and business-rule checks**

| Check | Count |
|---|---:|
| orphan_events (session_id not in sessions) | 150 |
| orphan_transactions (session_id not in sessions) | 5 |
| events_before_session_start | 0 |
| transactions_before_session_start | 6 |
| revenue_unparseable | 0 |
| revenue_negative | 8 |
| revenue_zero | 0 |
| test_transactions | 15 |
| sessions_with_purchase_before_checkout | 25 |
| sessions_with_transaction_but_no_purchase_event | 38 |
| sessions_with_purchase_event_but_no_valid_transaction | 8 |
| sessions_without_session_start_event | 690 |


### What the profile tells us

| Issue | Why it matters | Treatment |
|---|---|---|
| Mixed timestamp formats (`YYYY-MM-DD HH:MM:SS` and `…T…Z`) | Duplicates that differ only in format escape naive de-duplication | Parse both as UTC **before** de-duplicating |
| Label variants in source/medium/device/event/status/currency | One channel would be split into several groups; funnel events would be missed | Trim, lower-case, alias map (e.g. `ppc→cpc`, `smartphone→mobile`, `view_item→product_view`) |
| UTM query strings / trailing slashes in `landing_page` | Same page appears as many rows | Strip query string and trailing slash |
| Missing `campaign` | **Expected** for organic, direct and referral traffic | Fill `(not set)`; not an error |
| Missing `device_category` / `country` | Session is still valid traffic | Keep the session; label `unknown` / `(not set)` |
| Exact duplicate rows (all tables) | Inflate counts and revenue | Drop after standardisation |
| Orphan events / transactions | Cannot be attributed to a channel | Quarantine (kept in a separate file, not deleted) |
| Negative revenue, test orders, transaction before session start | Break business rules | Quarantine; do **not** guess a corrected value |
| Purchase event before checkout | Likely client clock skew | Keep, flag, exclude from ordered funnel stage only |
| Transaction without purchase event | Tracking loss | Keep revenue; transaction table is the revenue source of truth |

Missing `product_id` on non-product events (`page_view`, `begin_checkout`…) is structural and needs no treatment.

## 3. Cleaning decisions and their impact

The cleaning itself runs in `python/clean_data.py`; this notebook reads its log so the notebook and the pipeline cannot drift apart.

In [6]:
log = json.loads((MET / 'cleaning_log.json').read_text())
steps = pd.DataFrame(log['steps'])
steps[['table','step','rows_affected','decision']]

,table,step,rows_affected,decision
0,sessions,parse_timestamps,0,Parsed mixed ISO-8601 formats ('YYYY-MM-DD HH:...
1,sessions,standardise_labels,1076,"Trimmed, lower-cased and mapped aliases in `tr..."
2,sessions,standardise_labels,717,"Trimmed, lower-cased and mapped aliases in `ma..."
3,sessions,standardise_labels,893,"Trimmed, lower-cased and mapped aliases in `de..."
4,sessions,normalise_landing_page,1682,Removed query strings (e.g. ?utm_source=...) a...
5,sessions,fill_missing_device,107,Missing device_category set to 'unknown'; sess...
6,sessions,fill_missing_country,358,Missing country set to '(not set)'; sessions r...
7,sessions,fill_missing_campaign,20770,Missing campaign set to '(not set)'.
8,sessions,drop_exact_duplicates,40,Dropped rows identical after parsing/standardi...
9,sessions,drop_conflicting_duplicate_ids,0,Kept the earliest record for any session_id st...


In [7]:
print('Rows in model tables:', json.dumps(log['rows_out'], indent=1))
print('Quarantined:', log['quarantined'], '| quarantined positive revenue: $', log['quarantined_revenue'])
pd.read_csv(ROOT/'data/processed/quarantine_transactions.csv').quarantine_reason.value_counts()

Rows in model tables: {
 "fact_sessions": 35822,
 "fact_events": 160474,
 "fact_transactions": 913,
 "dim_date": 182,
 "dim_traffic_source": 17,
 "dim_device": 4,
 "dim_landing_page": 10,
 "dim_funnel_stage": 5
}
Quarantined: {'events': 150, 'transactions': 33} | quarantined positive revenue: $ 562.09


quarantine_reason
test_order              15
non_positive_revenue     8
before_session_start     5
orphan_session_id        5
Name: count, dtype: int64

## 4. Post-cleaning profile

In [8]:
clean_report = dq.run('processed')
display(Markdown(dq.to_markdown(clean_report)))

## Data-quality profile: `processed`

| Table | Rows | Exact duplicate rows | Duplicate PKs | Unparseable timestamps | Timestamp range |
|---|---:|---:|---:|---:|---|
| sessions | 35,822 | 0 | 0 | 0 | 2025-01-01 00:02:20 → 2025-06-30 23:58:16 |
| events | 160,474 | 0 | 0 | 0 | 2025-01-01 00:02:20 → 2025-07-01 00:00:33 |
| transactions | 913 | 0 | 0 | 0 | 2025-01-01 03:51:01 → 2025-06-30 22:54:01 |

**Missing values**

- sessions: none
- events: `product_id`: 115,444
- transactions: none

**Non-standard labels**

- none

**Relationship and business-rule checks**

| Check | Count |
|---|---:|
| orphan_events (session_id not in sessions) | 0 |
| orphan_transactions (session_id not in sessions) | 0 |
| events_before_session_start | 0 |
| transactions_before_session_start | 0 |
| revenue_unparseable | 0 |
| revenue_negative | 0 |
| revenue_zero | 0 |
| test_transactions | 0 |
| sessions_with_purchase_before_checkout | 25 |
| sessions_with_transaction_but_no_purchase_event | 13 |
| sessions_with_purchase_event_but_no_valid_transaction | 13 |
| sessions_without_session_start_event | 0 |


### Residual flags are expected and explained

* **Purchase before checkout (25 sessions):** retained and flagged `purchase_sequence_violation = 1`. These sessions do not reach the ordered funnel's purchase stage, but their valid transactions still count as revenue.
* **Transaction without purchase event:** tracking loss on the client. Revenue is taken from the transaction table, so nothing is lost.
* **Purchase event without valid transaction:** the transaction was quarantined (negative revenue or impossible timestamp).

These three items fully explain why the transaction-based *Purchasing Sessions* KPI differs from the event-based funnel *Purchase* stage (see notebook 03).

## 5. Validation against the generator's ground truth

Because the data is synthetic, the generator recorded the row counts *before* defects were injected. Cleaning should recover them exactly, apart from records that were deliberately corrupted beyond repair.

In [9]:
meta = json.loads((ROOT/'data/raw/generation_metadata.json').read_text())
truth, d = meta['generated_before_defects'], meta['injected_defects']
chk = pd.DataFrame({
  'expected': [truth['sessions'], truth['events'] - d['events_missing_purchase_event_with_transaction'],
               truth['transactions'] - d['transactions_negative_revenue'] - d['transactions_before_session_start']],
  'after_cleaning': [log['rows_out']['fact_sessions'], log['rows_out']['fact_events'], log['rows_out']['fact_transactions']]},
  index=['sessions','events','transactions'])
chk['match'] = chk.expected == chk.after_cleaning
chk

,expected,after_cleaning,match
sessions,35822,35822,True
events,160474,160474,True
transactions,913,913,True


All three tables reconcile exactly with the generator's ground truth, which confirms the cleaning rules neither drop valid records nor keep invalid ones. (On real data this check is not possible; the quarantine files exist so a human can review excluded rows instead.)